# Latent SOPE — 50 Rollouts

Scaled-up version of `latent_sope.ipynb` (Steps 0–7) with production-scale data collection.

| Parameter | `latent_sope.ipynb` | This notebook |
|-----------|--------------------|--------------|
| Oracle rollouts (K) | 10 | 50 |
| Offline rollouts (N) | 5 | 50 |
| Training epochs | 5 | 50 |
| Diffusion steps | 64 | 256 |
| Synthetic trajectories | 16 | 50 |

**Checkpoint reuse:** Each step checks for existing results and skips if found.
Set `FORCE_RERUN = True` to recompute everything from scratch.

**Estimated runtime (first run):** ~25 min total.
**Estimated runtime (cached):** ~2 min (Steps 5–7 only).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import torch

REPO_ROOT = Path("../").resolve()
sys.path.insert(0, str(REPO_ROOT))

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"Repo root: {REPO_ROOT}")

# --- Set to True to recompute all steps from scratch ---
FORCE_RERUN = False

# --- Common paths ---
POLICY_DIR = Path("../third_party/robomimic/diffusion_policy_trained_models/test")
policy_train_dirs = sorted([d for d in POLICY_DIR.glob("*") if d.is_dir()])
assert len(policy_train_dirs) > 0, "No trained policies found."
policy_train_dir = policy_train_dirs[-1]
print(f"Using policy from: {policy_train_dir}")

# --- Common hyperparams ---
K = 50           # oracle rollouts
N_ROLLOUTS = 50  # offline rollouts
EPOCHS = 50      # diffusion training epochs
HORIZON = 60
gamma = 1.0
BATCH_SIZE = 64
NUM_TRAJS = 50   # synthetic trajectories
obs_keys = ["object", "robot0_eef_pos", "robot0_eef_quat", "robot0_gripper_qpos"]

## Step 0: Ground Truth (Oracle Baseline)

50 rollouts for a tighter oracle estimate (~12 min).
Saves result to JSON; skips if already computed.

In [ ]:
import json

from src.latent_sope.robomimic_interface.checkpoints import (
    load_checkpoint,
    build_rollout_policy_from_checkpoint,
    build_env_from_checkpoint,
)
from src.latent_sope.robomimic_interface.rollout import rollout

oracle_path = policy_train_dir / "oracle_50.json"

if oracle_path.exists() and not FORCE_RERUN:
    with open(oracle_path) as f:
        oracle_data = json.load(f)
    oracle_value = oracle_data["oracle_value"]
    oracle_returns = oracle_data["returns"]
    print(f"Loaded cached oracle from {oracle_path}")
    print(f"  Oracle V^pi = {oracle_value:.3f} (std={np.std(oracle_returns):.3f}, K={len(oracle_returns)})")
else:
    ckpt = load_checkpoint(policy_train_dir.resolve(), ckpt_path="last.pth")
    policy = build_rollout_policy_from_checkpoint(ckpt, device=torch.device(device), verbose=False)
    env = build_env_from_checkpoint(ckpt, render=False, render_offscreen=True, verbose=False)

    oracle_returns = []
    for i in range(K):
        stats = rollout(policy=policy, env=env, horizon=HORIZON, render=False)
        oracle_returns.append(stats.total_reward)
        if (i + 1) % 10 == 0:
            print(f"  [{i+1}/{K}] running mean = {np.mean(oracle_returns):.3f}")

    oracle_value = float(np.mean(oracle_returns))

    with open(oracle_path, "w") as f:
        json.dump({"oracle_value": oracle_value, "returns": oracle_returns,
                    "K": K, "horizon": HORIZON, "gamma": gamma}, f, indent=2)
    print(f"Saved oracle to {oracle_path}")

print(f"\nOracle V^pi over {len(oracle_returns)} rollouts (horizon={HORIZON}, gamma={gamma}):")
print(f"  mean = {oracle_value:.3f}, std = {np.std(oracle_returns):.3f}")

## Step 1: Collect Offline Data

50 rollouts (~12 min). Saves to `rollout_latents_50/`. Skips if directory already has N_ROLLOUTS files.

In [ ]:
from src.latent_sope.robomimic_interface.rollout import (
    save_rollout_latents,
    load_rollout_latents,
    get_policy_frame_stack,
    PolicyFeatureHook,
    RolloutLatentRecorder,
)

output_dir = policy_train_dir / "rollout_latents_50"
output_dir.mkdir(exist_ok=True)
existing_rollouts = sorted(output_dir.glob("*.h5"))

if len(existing_rollouts) >= N_ROLLOUTS and not FORCE_RERUN:
    rollout_paths = existing_rollouts[:N_ROLLOUTS]
    print(f"Found {len(existing_rollouts)} existing rollouts in {output_dir} — skipping collection.")
else:
    # Need policy + env for collection (load if not already loaded)
    if "ckpt" not in dir():
        ckpt = load_checkpoint(policy_train_dir.resolve(), ckpt_path="last.pth")
    if "policy" not in dir():
        policy = build_rollout_policy_from_checkpoint(ckpt, device=torch.device(device), verbose=False)
    if "env" not in dir():
        env = build_env_from_checkpoint(ckpt, render=False, render_offscreen=True, verbose=False)

    frame_stack = get_policy_frame_stack(policy)

    for i in range(N_ROLLOUTS):
        save_path = output_dir / f"rollout_{i:03d}.h5"
        if save_path.exists() and not FORCE_RERUN:
            continue

        feature_hook = PolicyFeatureHook(policy, feat_type="low_dim_concat")
        recorder = RolloutLatentRecorder(
            feature_hook, obs_keys=obs_keys, store_obs=True, store_next_obs=False,
        )
        stats = rollout(policy=policy, env=env, horizon=HORIZON, render=False, recorder=recorder)
        traj = recorder.finalize(stats)
        save_rollout_latents(save_path, traj)
        feature_hook.close()

        if (i + 1) % 10 == 0:
            print(f"  [{i+1}/{N_ROLLOUTS}] reward={stats.total_reward:.1f}, latents={traj.latents.shape}")

    rollout_paths = sorted(output_dir.glob("*.h5"))[:N_ROLLOUTS]
    print(f"Collected {len(rollout_paths)} rollout files in {output_dir}")

print(f"Using {len(rollout_paths)} rollout files")

## Step 2: Chunk the Offline Data

50 rollouts x ~25 chunks each = ~1250 chunks → ~19 batches of 64.
Skipped when loading diffusion model from checkpoint (Step 3 restores configs).

In [ ]:
from src.latent_sope.robomimic_interface.dataset import (
    RolloutChunkDatasetConfig,
    make_rollout_chunk_dataloader,
)

ckpt_path_check = policy_train_dir / "diffusion_ckpts_50" / "sope_diffuser_latest.pt"
need_chunking = not ckpt_path_check.exists() or FORCE_RERUN

if need_chunking:
    sample_traj = load_rollout_latents(rollout_paths[0])
    latents_dim = sample_traj.latents.shape[-1]
    action_dim = sample_traj.actions.shape[-1]
    print(f"Latent dim: {latents_dim}, Action dim: {action_dim}")

    dataset_config = RolloutChunkDatasetConfig(
        chunk_size=8,
        stride=2,
        frame_stack=2,
        source="latents",
        latents_dim=latents_dim,
        action_dim=action_dim,
        normalize=True,
        return_metadata=True,
    )

    dataloader, norm_stats = make_rollout_chunk_dataloader(
        paths=rollout_paths,
        config=dataset_config,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=True,
    )

    print(f"DataLoader: {len(dataloader)} batches of size {BATCH_SIZE}")
    if norm_stats is not None:
        print(f"Normalization stats: mean shape={norm_stats.mean.shape}")

    batch = next(iter(dataloader))
    for k, v in batch.items():
        if isinstance(v, torch.Tensor):
            print(f"  {k}: {v.shape} {v.dtype}")
        else:
            print(f"  {k}: {type(v).__name__}")
else:
    print("Diffusion checkpoint exists — skipping chunking (configs will be restored in Step 3).")

## Step 3: Train Chunk Diffusion

256 diffusion steps (SOPE default), 50 epochs. ~5 min estimated.
Loads from checkpoint if available; otherwise trains and saves.

In [ ]:
from tqdm.auto import tqdm
from dataclasses import asdict
from src.latent_sope.diffusion.sope_diffuser import (
    SopeDiffusionConfig,
    SopeDiffuser,
    NormalizationStats as DiffusionNormStats,
    cross_validate_configs,
)

ckpt_dir = policy_train_dir / "diffusion_ckpts_50"
ckpt_path = ckpt_dir / "sope_diffuser_latest.pt"

if ckpt_path.exists() and not FORCE_RERUN:
    # ─── Load from checkpoint ───
    print(f"Loading diffusion checkpoint from {ckpt_path}")
    ckpt_payload = torch.load(str(ckpt_path), map_location=device, weights_only=False)

    diffusion_config = SopeDiffusionConfig(**ckpt_payload["diffusion_config"])
    dataset_config = RolloutChunkDatasetConfig(**ckpt_payload["dataset_config"])

    diff_norm_stats = None
    if ckpt_payload["normalization_stats"] is not None:
        ns = ckpt_payload["normalization_stats"]
        diff_norm_stats = DiffusionNormStats(mean=ns["mean"], std=ns["std"])
        norm_stats = ns  # keep for reference

    diffuser = SopeDiffuser(cfg=diffusion_config, normalization_stats=diff_norm_stats, device=device)
    diffuser.diffusion.load_state_dict(ckpt_payload["diffusion_state_dict"])
    diffuser.diffusion.eval()

    latents_dim = diffusion_config.state_dim
    action_dim = diffusion_config.action_dim
    EPOCHS = ckpt_payload["epoch"]

    print(f"  Loaded: {EPOCHS} epochs, {ckpt_payload['step']} steps")
    print(f"  state_dim={latents_dim}, action_dim={action_dim}, diffusion_steps={diffusion_config.diffusion_steps}")
else:
    # ─── Train from scratch ───
    diffusion_config = SopeDiffusionConfig(
        chunk_horizon=dataset_config.chunk_size,
        frame_stack=dataset_config.frame_stack,
        state_dim=latents_dim,
        action_dim=action_dim,
        diffusion_steps=256,
        dim_mults=(1, 2),
        attention=False,
        loss_type="l2",
        action_weight=5.0,
        predict_epsilon=True,
        lr=3e-4,
        guided=False,
    )

    cross_validate_configs(dataset_config, diffusion_config)
    print("Config cross-validation passed.")

    diff_norm_stats = None
    if norm_stats is not None:
        diff_norm_stats = DiffusionNormStats(mean=norm_stats.mean, std=norm_stats.std)

    diffuser = SopeDiffuser(cfg=diffusion_config, normalization_stats=diff_norm_stats, device=device)
    optimizer = diffuser.make_optimizer()

    n_params = sum(p.numel() for p in diffuser.diffusion.parameters())
    print(f"TemporalUnet parameters: {n_params:,}")
    print(f"Training for {EPOCHS} epochs...")

    # --- Training loop ---
    import matplotlib.pyplot as plt

    GRAD_CLIP = 1.0
    losses = []
    diffuser.diffusion.train()

    for epoch in range(1, EPOCHS + 1):
        epoch_losses = []
        for batch in tqdm(dataloader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False):
            batch_dev = {
                k: v.to(device) if isinstance(v, torch.Tensor) else v
                for k, v in batch.items()
            }
            loss, info = diffuser.loss(batch_dev)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(diffuser.diffusion.parameters(), GRAD_CLIP)
            optimizer.step()
            losses.append(loss.item())
            epoch_losses.append(loss.item())

        if epoch % 10 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}: mean loss = {np.mean(epoch_losses):.4f}")

    plt.figure(figsize=(10, 3))
    plt.plot(losses, alpha=0.3, color="steelblue")
    window = min(50, max(1, len(losses) // 5))
    if window > 1:
        smoothed = np.convolve(losses, np.ones(window) / window, mode="valid")
        plt.plot(range(window - 1, len(losses)), smoothed, color="steelblue", linewidth=2)
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.title(f"Chunk diffusion training loss ({EPOCHS} epochs, {len(rollout_paths)} rollouts)")
    plt.tight_layout()
    plt.show()

    # --- Save checkpoint ---
    ckpt_dir.mkdir(exist_ok=True)
    ckpt_payload = {
        "diffusion_state_dict": diffuser.diffusion.state_dict(),
        "epoch": EPOCHS,
        "step": len(losses),
        "diffusion_config": asdict(diffusion_config),
        "dataset_config": asdict(dataset_config),
        "normalization_stats": {
            "mean": norm_stats.mean, "std": norm_stats.std,
        } if norm_stats is not None else None,
    }
    torch.save(ckpt_payload, str(ckpt_path))
    print(f"Saved checkpoint to {ckpt_path}")
    print(f"  Final loss: {losses[-1]:.4f}")

## Step 5: Generate Synthetic Trajectories via Stitching

Generate full-length trajectories by autoregressively stitching diffusion chunks.

In [ ]:
NUM_TRAJS = 50
MAX_LENGTH = HORIZON

# Sample initial states from offline rollouts
init_states = []
for i in range(NUM_TRAJS):
    traj_i = load_rollout_latents(rollout_paths[i % len(rollout_paths)])
    latents = traj_i.latents
    if latents.ndim == 3:
        latents = latents[:, 0, :]
    init_states.append(latents[0])

init_states_t = torch.tensor(np.stack(init_states), dtype=torch.float32)

print(f"Generating {NUM_TRAJS} stitched trajectories (max {MAX_LENGTH} steps)...\n")

syn_states, syn_actions, end_indices = diffuser.generate_full_trajectory(
    initial_states=init_states_t,
    max_length=MAX_LENGTH,
    guided=False,
    verbose=False,
)

print(f"Generated {NUM_TRAJS} trajectories:")
print(f"  states shape: {syn_states.shape}")
print(f"  actions shape: {syn_actions.shape}")
print(f"  state range: [{syn_states.min():.2f}, {syn_states.max():.2f}]")

## Step 6: Score Trajectories (Ground-Truth Reward)

Score synthetic trajectories using the analytical Lift reward function:
`cube_z > table_height + 0.04` → reward 1.0, else 0.0.

Decodes latents → obs dict via `LowDimConcatEncoder`, then checks cube z-position directly.

In [ ]:
from src.latent_sope.eval.reward_model import LiftRewardFn, score_trajectories_gt, make_lift_encoder

# Build encoder configured for Lift obs decoding (single-frame dims, not frame_stack)
encoder = make_lift_encoder(obs_keys=obs_keys)

# Ground-truth reward: cube z > 0.84
reward_fn = LiftRewardFn(table_height=0.8, height_threshold=0.04)
print(f"Reward function: {reward_fn}")

synthetic_returns, synthetic_rewards = score_trajectories_gt(
    reward_fn=reward_fn,
    encoder=encoder,
    states=syn_states,
    actions=syn_actions,
    gamma=gamma,
)

print(f"\nSynthetic trajectory returns (gamma={gamma}):")
print(f"  mean = {synthetic_returns.mean():.3f}, std = {synthetic_returns.std():.3f}")
print(f"  per-trajectory (first 10): {[f'{r:.2f}' for r in synthetic_returns[:10]]}")

# Diagnostic: cube z-positions in generated trajectories
obs_decoded = encoder.decode_to_obs_dict(syn_states[0])
cube_z = obs_decoded["object"][:, 2]
print(f"\nDiagnostic — trajectory 0 cube z-positions:")
print(f"  range: [{cube_z.min():.4f}, {cube_z.max():.4f}]")
print(f"  success threshold: {reward_fn.success_z:.4f}")
print(f"  steps above threshold: {(cube_z > reward_fn.success_z).sum()} / {len(cube_z)}")

## Step 7: Evaluate the OPE Estimate

Compare OPE estimate to oracle ground truth.

In [ ]:
from src.latent_sope.eval.metrics import ope_eval

result = ope_eval(oracle_value, synthetic_returns)

print("=" * 50)
print("OPE Evaluation (50 rollouts)")
print("=" * 50)
print(f"  Oracle V^pi:      {result.oracle_value:.3f}")
print(f"  OPE estimate:     {result.ope_estimate:.3f} (std={result.ope_std:.3f})")
print(f"  MSE:              {result.mse:.6f}")
print(f"  Relative error:   {result.relative_error:.2%}")
print("=" * 50)